# TrustLens — end-to-end Colab notebook

Detects scam job postings using a fine-tuned Phi-3-mini, a LangChain ReAct agent, ChromaDB for kNN risk features, and SQLite for prediction logging.

## What this notebook does (in order)
1. **Setup** — install pinned deps, clone the GitHub repo, configure the HF token.
2. **Data** — verify the train/val/test splits the EDA notebook produced locally.
3. **ChromaDB index** — embed the training postings (one-shot, ~3 min on T4).
4. **Heuristic features** — show the 7 cheap signals.
5. **Teacher labelling** — small demo run with Llama-3.3-70B via HF API.
6. **Baselines** — train DistilBERT (B4) and run Llama few-shot (B3) on the val set.
7. **QLoRA fine-tuning** — Phi-3-mini + LoRA rank-16 on q/k/v/o, 1 epoch, ~30–60 min on T4.
8. **Evaluation** — F1 / MAE / Pearson r / ROUGE-L / JSON-validity comparison table.
9. **Inference demo** — paste a posting, get a structured trust assessment.

**Runtime:** ~1.5–2 hours on Colab Free T4 if you run the fine-tuning cell. ~15 min if you skip it.

**Before you run:** Runtime → Change runtime type → **T4 GPU**.

## 1. Setup

In [ ]:
!pip install -q transformers==4.45.2 peft==0.13.2 bitsandbytes==0.44.1 accelerate==1.0.1 datasets==3.0.2 \
    langchain==0.3.7 langchain-community==0.3.5 langchain-huggingface huggingface_hub \
    sentence-transformers==3.2.1 chromadb==0.5.15 \
    rouge-score==0.1.2 python-dotenv==1.0.1 plotly==5.24.1

In [ ]:
# Clone the project repo. Replace <YOUR-USERNAME> with your GitHub handle.
import os
GITHUB_USER = '<YOUR-USERNAME>'
REPO = 'trustlens'

if not os.path.exists(REPO):
    !git clone https://github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}
!ls

In [ ]:
# Put your HF token in Colab Secrets (Tools -> Secrets) as 'HF_TOKEN'.
# This cell pulls it from there into the env. Falls back to interactive input.
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    from getpass import getpass
    os.environ['HF_TOKEN'] = getpass('paste HF_TOKEN: ')
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

In [ ]:
# Optional: mount Drive for persisting the fine-tuned LoRA adapter.
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/trustlens'
os.makedirs(DRIVE_DIR, exist_ok=True)

## 2. Data

The EDA + stratified split was done locally in `notebooks/01_eda.ipynb`. The split CSVs are gitignored (too big), so we regenerate them here from the raw Kaggle CSV.

**Upload `aligned_kaggle_data.csv` to `data/raw/` from the Colab file explorer (left sidebar) before running this cell.**

In [ ]:
import os, re
import pandas as pd
from sklearn.model_selection import train_test_split

RAW = 'data/raw/aligned_kaggle_data.csv'
PROC = 'data/processed'
os.makedirs(PROC, exist_ok=True)

df = pd.read_csv(RAW)
df['fraudulent'] = df['fraudulent'].astype(int)
PRE = ['desc_length','req_length','profile_length','kw_work_from_home','kw_quick_money',
       'kw_no_experience','kw_urgent','kw_immediate_start','desc_missing','profile_missing']
df = df.drop(columns=[c for c in PRE if c in df.columns])
df = df[df['description'].fillna('').str.strip() != '']
df = df.drop_duplicates(subset=['description'], keep='first')

EMAIL_RE = re.compile(r'(\b[\w.+-]+@)([\w.-]+\.\w+)\b')
for col in ['title','company_profile','description','requirements','benefits']:
    df[col] = df[col].apply(lambda t: EMAIL_RE.sub(lambda m: m.group(1)+m.group(2).lower(), t) if isinstance(t,str) else t)

TAGS = [('title','TITLE'),('location','LOCATION'),('employment_type','EMPLOYMENT TYPE'),
        ('industry','INDUSTRY'),('company_profile','COMPANY PROFILE'),('description','DESCRIPTION'),
        ('requirements','REQUIREMENTS'),('benefits','BENEFITS'),('salary_range','SALARY')]
def build_text(row):
    parts = [f'[{t}] {row[c].strip()}' for c,t in TAGS if isinstance(row.get(c),str) and row[c].strip()]
    return '\n'.join(parts)
df['job_text'] = df.apply(build_text, axis=1)

tr, tmp = train_test_split(df, test_size=0.20, stratify=df['fraudulent'], random_state=42)
va, te = train_test_split(tmp, test_size=0.50, stratify=tmp['fraudulent'], random_state=42)
for n, p in [('train',tr),('val',va),('test',te)]:
    p.to_csv(f'{PROC}/{n}.csv', index=False)
    print(f'{n}: {len(p)} rows, fraud rate {p["fraudulent"].mean():.2%}')

## 3. Build the ChromaDB index

Embeds the training postings into a persistent vector store. Used for the kNN `neighbor_fraud_rate` feature and the UI similarity panel — **never** for prompt augmentation. This is intentionally not RAG.

Takes ~3 min on T4 (most of it is the all-MiniLM-L6-v2 download + batch embed).

In [ ]:
!python -m scripts.build_chroma_index

## 4. Heuristic features (sanity check)

The 7 cheap signals from `src/features.py` that get passed to the LLM as a numeric block.

In [ ]:
from src.features import extract_all

scam = '[TITLE] Earn $5000/week from home!\n[DESCRIPTION] URGENT immediate hire, apply now! Email recruiter@gmail.com to start. Pay registration fee.'
real = '[TITLE] Senior Software Engineer\n[COMPANY PROFILE] Stripe is a payments infrastructure company.\n[DESCRIPTION] We are hiring backend engineers in San Francisco.'
print('SCAM:', extract_all(scam))
print()
print('REAL:', extract_all(real))

## 5. Teacher labelling (small demo)

Runs `src/label_generator.py --dry-run` to label 5 real postings via Llama-3.3-70B-Instruct on the HF Inference API.

For the full 1000-row run that produces fine-tuning data, drop `--dry-run` (takes ~30+ min on free tier).

In [ ]:
!python -m src.label_generator --dry-run

In [ ]:
# Uncomment for the full job (writes data/labeled/train.jsonl).
# Required before STEPs 6 (B2/B3 baselines) and 7 (fine-tuning).
# !python -m src.label_generator --n 1000

## 6. Baselines

Four baselines for the rubric's improvement-over-baselines requirement. We run B4 (DistilBERT, ~5 min on T4) and B3 (Llama-3.3-70B few-shot via HF API) here. B1 and B2 (Phi-3 zero-shot / few-shot) are commented out — uncomment if you want to run them too.

In [ ]:
# B4: DistilBERT classifier (non-LLM baseline). Fast on T4.
!python -m src.baselines --which b4

In [ ]:
# B3: Llama-3.3-70B few-shot via HF Inference API. Requires train.jsonl from STEP 5.
# !python -m src.baselines --which b3

In [ ]:
# B1 + B2: Phi-3-mini zero-shot and few-shot. Heavier — uncomment to run.
# !python -m src.baselines --which b1
# !python -m src.baselines --which b2

## 7. QLoRA fine-tuning of Phi-3-mini

**Skip this section if you only want the demo.** It takes ~30–60 min on T4 and requires `data/labeled/train.jsonl` to exist (run cell `teacher-full` first).

Loads `microsoft/Phi-3-mini-4k-instruct` in 4-bit (NF4 + bf16 compute), attaches a LoRA adapter (rank 16, alpha 32, q/k/v/o projections, dropout 0.05), trains for 1 epoch with `paged_adamw_8bit` and gradient checkpointing, saves the adapter to Drive.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL = 'microsoft/Phi-3-mini-4k-instruct'

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
tok_ft = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tok_ft.pad_token is None:
    tok_ft.pad_token = tok_ft.eos_token
mdl_ft = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb,
                                               device_map='auto', trust_remote_code=True)
mdl_ft = prepare_model_for_kbit_training(mdl_ft)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                  target_modules=['q_proj','k_proj','v_proj','o_proj'],
                  bias='none', task_type='CAUSAL_LM')
mdl_ft = get_peft_model(mdl_ft, lora)
mdl_ft.print_trainable_parameters()

In [ ]:
import json
from datasets import Dataset
from src.label_generator import SYSTEM_PROMPT, build_user_prompt

LABEL_KEYS = ['trust_score', 'red_flags', 'risk_breakdown', 'action', 'reasoning']
TRAIN_JSONL = 'data/labeled/train.jsonl'

records = []
with open(TRAIN_JSONL) as f:
    for line in f:
        r = json.loads(line)
        feats = {k.replace('feat_',''): v for k,v in r.items() if k.startswith('feat_')}
        records.append({'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': build_user_prompt(r['job_text'], feats, r['neighbor_fraud_rate'])},
            {'role': 'assistant', 'content': json.dumps({k: r[k] for k in LABEL_KEYS})},
        ]})

def to_features(ex):
    text = tok_ft.apply_chat_template(ex['messages'], tokenize=False)
    out = tok_ft(text, truncation=True, max_length=2048, padding=False)
    out['labels'] = out['input_ids'].copy()
    return out

ds = Dataset.from_list(records).map(to_features, remove_columns=['messages'])
print(f'{len(ds)} training examples')

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

ADAPTER_OUT = f'{DRIVE_DIR}/phi3-trustlens-lora'

args = TrainingArguments(
    output_dir='/content/checkpoints',
    num_train_epochs=1, per_device_train_batch_size=2, gradient_accumulation_steps=8,
    learning_rate=2e-4, warmup_ratio=0.1, gradient_checkpointing=True,
    fp16=False, bf16=True, logging_steps=20, save_steps=200, save_total_limit=2,
    report_to='none', optim='paged_adamw_8bit',
)
trainer = Trainer(model=mdl_ft, args=args, train_dataset=ds,
                  data_collator=DataCollatorForLanguageModeling(tok_ft, mlm=False))
trainer.train()
mdl_ft.save_pretrained(ADAPTER_OUT)
tok_ft.save_pretrained(ADAPTER_OUT)
print(f'adapter saved to {ADAPTER_OUT}')

## 8. Evaluation

Reads every baseline JSONL plus the fine-tuned model's predictions (if present) and writes a markdown comparison table + bar chart to `results/`.

In [ ]:
!python -m src.evaluate

In [ ]:
from IPython.display import Markdown, Image
Markdown(open('results/comparison_table.md').read())

In [ ]:
Image('results/comparison.png')

## 9. Inference demo

Paste any job posting text and get a structured trust assessment back. The agent calls the 4 LangChain tools (heuristic features, company allowlist, salary realism, email-domain class) before writing the JSON Final Answer.

In [ ]:
from src.pipeline import analyze_job

POSTING = '''[TITLE] Customer Service Representative — Work From Home
[COMPANY PROFILE] We are a fast growing online business.
[DESCRIPTION] URGENT! Earn $4500 per week working from home. No experience needed!
Apply now to start tomorrow. Send your details to recruiter@gmail.com and pay a one-time
$50 starter kit fee to get your training materials. Limited spots available!'''

result = analyze_job(POSTING)
import json
print(json.dumps(result, indent=2))

In [ ]:
POSTING_REAL = '''[TITLE] Senior Backend Engineer
[LOCATION] San Francisco, CA
[COMPANY PROFILE] Stripe builds economic infrastructure for the internet.
[DESCRIPTION] We are looking for a backend engineer to join our payments platform team.
You will work with Go, Postgres, and distributed systems. Minimum 4 years of experience.
[SALARY] 180000-260000
[REQUIREMENTS] Strong CS fundamentals, experience with distributed systems.'''

result = analyze_job(POSTING_REAL)
print(json.dumps(result, indent=2))

## What lives outside this notebook

- **Streamlit UI** — `app.py` runs locally on your Mac (`streamlit run app.py`). Same `analyze_job` backend.
- **Error analyzer** — `python -m src.error_analyzer` walks misclassified val rows.
- **Study sheets** — `docs/study_sheet_*.md`, one per non-trivial source file, with line-by-line walkthroughs and viva Q&A.
- **README** — architecture overview, rubric mapping, why-each-choice.